In [0]:
dbutils.widgets.text('landing_timestamp', "")
dbutils.widgets.text('table_name', "")
dbutils.widgets.text('merge_keys', "")
dbutils.widgets.text('load_type', "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")
print("=" * 60)
print("STEP 1: Widget Parameters Received")
print("=" * 60)
print(f"  table_name         : {table_name}")
print(f"  landing_timestamp  : {landing_timestamp}")
print(f"  load_type          : {load_type}")
print(f"  merge_keys         : {merge_keys}")
print("=" * 60)

In [0]:
bronze_df = spark.read.table(f"caresync_databricks_workspace.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

row_count = bronze_df.count()
col_count = len(bronze_df.columns)

print("=" * 60)
print(f"STEP 2: Bronze Read — caresync_databricks_workspace.bronze.{table_name}")
print("=" * 60)
print(f"  Filter applied     : landing_timestamp = '{landing_timestamp}'")
print(f"  Rows loaded        : {row_count:,}")
print(f"  Columns            : {col_count}")
print(f"  Column names       : {bronze_df.columns}")
print("-" * 60)
print("  Schema:")
bronze_df.printSchema()
print("-" * 60)
print("  Sample rows (up to 15):")
bronze_df.show(15, truncate=False)

In [0]:
from pyspark.sql import functions as F

updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("=" * 60)
print("STEP 3: Audit Columns Added")
print("=" * 60)
print("  Added columns      : insert_timestamp, update_timestamp")
print(f"  Final column count : {len(updated_df.columns)}")
print(f"  Final columns      : {updated_df.columns}")
print("-" * 60)
print("  Sample rows with audit columns (up to 10):")
updated_df.select("insert_timestamp", "update_timestamp", *[c for c in updated_df.columns if c not in ('insert_timestamp','update_timestamp')][:5]).show(10, truncate=False)

In [0]:
target_table = f"caresync_databricks_workspace.silver.{table_name}"

print("=" * 60)
print(f"STEP 4: Write to Silver — {target_table}")
print("=" * 60)
print(f"  Load type          : {load_type.upper()}")

if load_type.upper() == "FULL":
    print("  Strategy           : OVERWRITE (full replace with schema evolution)")
    updated_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    print("  ✅ Full overwrite complete.")

elif load_type.upper() == "APPEND":
    print("  Strategy           : APPEND (with schema merge)")
    updated_df.write.mode("append").option("mergeSchema", "true").saveAsTable(target_table)
    print("  ✅ Append complete.")

elif load_type.upper() == "MERGE":
    # First run: target table may not exist yet — create it with an overwrite
    if not spark.catalog.tableExists(target_table):
        print(f"  Target table does not exist yet — creating via overwrite.")
        updated_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
        print("  ✅ Initial table created (first-run overwrite).")
    else:
        keys = [k.strip() for k in merge_keys.split(",")]
        print(f"  Strategy           : MERGE (upsert)")
        print(f"  Merge keys         : {keys}")

        updated_df.createOrReplaceTempView("_merge_source")

        merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

        update_cols = [c for c in updated_df.columns if c != "insert_timestamp"]
        update_set  = ", ".join([f"{c} = source.{c}" for c in update_cols])

        all_cols = ", ".join(updated_df.columns)
        all_vals = ", ".join([f"source.{c}" for c in updated_df.columns])

        merge_sql = (
            f"MERGE INTO {target_table} AS target "
            f"USING _merge_source AS source "
            f"ON {merge_condition} "
            f"WHEN MATCHED THEN UPDATE SET {update_set} "
            f"WHEN NOT MATCHED THEN INSERT ({all_cols}) VALUES ({all_vals})"
        )
        print(f"  Merge SQL:\n    {merge_sql}")
        spark.sql(merge_sql)
        print("  ✅ Merge complete.")

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected: FULL, APPEND, or MERGE.")

# Post-write verification
silver_df = spark.read.table(target_table)
silver_count = silver_df.count()
print("-" * 60)
print(f"  Silver table row count after write : {silver_count:,}")
print("  Silver sample rows (up to 10):")
silver_df.show(10, truncate=False)
print("=" * 60)

In [0]:
record_count = updated_df.count()

print("=" * 60)
print("STEP 5: Notebook Exit")
print("=" * 60)
print(f"  Records processed  : {record_count:,}")
print(f"  Returning value    : {record_count}")
print("=" * 60)

dbutils.notebook.exit(str(record_count))